# Implementing a web search tool

## Preparations

Sign up at https://www.tavily.com, get an API key and add it to the environment. Then, install `tavily-pytion` client.

## Verify basic web searching function using Tavily

We verify that we can search the web using the `TavilyClient`:


In [ ]:
from tavily import TavilyClient

tavily_client = TavilyClient()

In [2]:
def search_web(query: str, max_results: int = 2) -> list:
  response = tavily_client.search(query, max_results=max_results)
  return response.get("results")

Lets search for Kipchoge's marathon record to verify search_web.

In [3]:
search_web("Kipcoge's marathon world record")

[{'url': 'https://coroscom.wpcomstaging.com/world-record/',
  'title': "Kipchoge's World Record: Inside The Numbers - COROS Stories",
  'content': '40-42.195km: 6:16 (2:01:09) ... The best marathon performance of all time took place on September 25th, 2022 in Berlin Germany. Surrounded by a',
  'score': 0.9997918,
  'raw_content': None},
 {'url': 'https://en.wikipedia.org/wiki/Eliud_Kipchoge',
  'title': 'Eliud Kipchoge - Wikipedia',
  'content': 'On 16 September, Kipchoge won the 2018 Berlin Marathon in a time of 2:01:39, breaking the previous world record by 1 minute and 18 seconds (2:02:57 set by fellow countryman Dennis Kimetto at the Berlin Marathon in 2014). On 20 January, Kipchoge announced his desire to win all six World Marathon Majors (he had already won three, the London, Berlin, and Chicago marathons by that time). On 25 September, Kipchoge won the Berlin Marathon decisively in a time of 2:01:09, beating by 30 seconds his own previous world record, which he set on the same 

### Adding search options

We expand our search function with several options:

- `topic` specifies a content type like: general, news or finance.
- `time_range` filters results by recency, when we need current vs historical information
- `country` prefers search results from a specific country

See https://docs.tavily.com/documentation/api-reference/endpoint/search for more information.

> The more options a tool exposes, the more complex your tool definitions becomes, and the harder it is for the LLM to use it correctly.


In [4]:
def search_web(
    query: str,
    max_results: int = 2,
    topic: str = "general",
    time_range: str | None = None,
    country: str | None = None
) -> list:
  
  response = tavily_client.search(
    query,
    max_results=max_results,
    topic=topic,
    time_range=time_range,
    country=country
  )
  return response.get("results")

### Handle errors gracefully

In the following example, we handle errors with a catch-all approach. In production, different errors require different handling:

- 401 Authentication errors indicate configuration problems
- 429 Rate limit errors benefit from retry with backoff
- Timeout errors might need a simpler query
- Network errors are different from API errors

In [5]:
def search_web(
    query: str,
    max_results: int = 2,
    topic: str = "general",
    time_range: str | None = None,
    country: str | None = None
) -> list | str:
    """Search the web for the given query."""
  
    try:
        response = tavily_client.search(
            query,
            max_results=max_results,
            topic=topic,
            time_range=time_range,
            country=country
        )
        return response.get("results")
    
    except Exception as e:
        return f"Error: Search failed - {e}"


### Converting to tool definitions

To use our tools in combination with an LLM, we need to define them in a standardized tool definition format. Functions may change frequently, so manual updating these definitions is error-prone and inefficient. So we generate these definitions automatically by converting Python functions into a tool definition.

We use the `inspect`module to extract:
- functions name
- docstring
- parameter details

In [6]:
import inspect
 
def example_tool(input_1:str, input_2:int=1):
    """docstring for example_tool"""
    return
        
print(f"function name: {example_tool.__name__}")
print(f"function docstring: {example_tool.__doc__}")
print(f"function signature: {inspect.signature(example_tool)}")

function name: example_tool
function docstring: docstring for example_tool
function signature: (input_1: str, input_2: int = 1)


In [7]:
def function_to_input_schema(func) -> dict:
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }
    
    try:
        signature = inspect.signature(func)
    except ValueError as e:
        raise ValueError(
            f"Failed to get signature for function {func.__name__}: {str(e)}"
        )
    
    parameters = {}
    for param in signature.parameters.values():
        try:
            param_type = type_map.get(param.annotation, "string")
        except KeyError as e:
            raise KeyError(
                f"Unknown type annotation {param.annotation} for parameter {param.name}: {str(e)}"
            )
        parameters[param.name] = {"type": param_type}
    
    required = [
        param.name
        for param in signature.parameters.values()
        if param.default == inspect._empty
    ]
    
    return {
            "type": "object",
            "properties": parameters,
            "required": required,
        }

The signature is used to extract function parameters and convert their types. If a parameter has no default value, it is required. Lets try to convert our web search tool:

In [8]:
def format_tool_definition(name: str, description: str, parameters: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }
 
def function_to_tool_definition(func) -> dict:
    return format_tool_definition(
        func.__name__,
        func.__doc__ or "",
        function_to_input_schema(func)
    )
 
search_tool_definition = function_to_tool_definition(search_web)
print(search_tool_definition)



{'type': 'function', 'function': {'name': 'search_web', 'description': 'Search the web for the given query.', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'max_results': {'type': 'integer'}, 'topic': {'type': 'string'}, 'time_range': {'type': 'string'}, 'country': {'type': 'string'}}, 'required': ['query']}}}


### End-to-end tool execution

We put everything together with our tool execution system. For this, we combine:

- a _toolbox_ that maps tool names to functions
- the _tool call_ from our LLM to be executed with the provided arguments

We return the result from the tool.

In [9]:
import json

def tool_execution(tool_box, tool_call):
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    return tool_box[function_name](**function_args)

From this we build a *control loop* to manage the interaction between LLM and the tools. These are the steps:

1. The LLM is given the _system prompt_, _user question_ and _tool definitions_.
2. The LLM determines what to do. If it needs external information, a tool call is generated.
3. The LLM returns is message, that is appended as _assistant_ message to the conversation
4. Now, we execute the requested tool and append its answer as _tool_ message to the history
5. We repeat this until the LLM has all information needed. This is when it responds without a tool call.

In [ ]:
from litellm import completion

tools = [search_web]
model = "ollama/gpt-oss:120b-cloud"

def simple_agent_loop(system_prompt, question):
    tool_box = {tool.__name__: tool for tool in tools}
    tool_definitions = [function_to_tool_definition(tool) for tool in tools]
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    while True:
        response = completion(
            model=model,
            messages=messages,
            tools=tool_definitions
        )

        assistant_message = response.choices[0].message
        
        if assistant_message.tool_calls:
            messages.append(assistant_message)
            for tool_call in assistant_message.tool_calls:
                tool_result = tool_execution(tool_box, tool_call)
                messages.append({
                    "role": "tool",
                    "content": str(tool_result),
                    "tool_call_id": tool_call.id
                })
        else:
            return assistant_message.content


In [30]:
system_prompt = """You are a helpful assistant. 
Use the search tool when you need current information."""
 
result = simple_agent_loop(
    system_prompt, 
    "Who won the 2025 Nobel Prize in Physics?"
)
print(result)

KeyboardInterrupt: 